In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

CATALOG = "workspace"
SCHEMA = "incident_assistent"     
LLM = "databricks-meta-llama-3-3-70b-instruct"

INDEX_A = f"{CATALOG}.{SCHEMA}.idx_small"
INDEX_B = f"{CATALOG}.{SCHEMA}.idx_large"
eval_table = f"{CATALOG}.{SCHEMA}.eval_questions"

w = WorkspaceClient()

def search(question, index_name, k=3):
    res = w.vector_search_indexes.query_index(
        index_name=index_name,
        columns=["chunk_id", "doc_id", "text"],
        query_text=question,
        num_results=k,
    )
    return res.result.data_array or []

def call_llm(prompt, max_tokens=400):
    resp = w.serving_endpoints.query(
        name=LLM,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return resp.choices[0].message.content

for n in [INDEX_A, INDEX_B]:
    print(n, "-> ready:", w.vector_search_indexes.get_index(n).status.ready)

workspace.incident_assistent.idx_small -> ready: True
workspace.incident_assistent.idx_large -> ready: True


In [0]:
import time

docs = spark.table(f"{CATALOG}.{SCHEMA}.bronze_docs").select("doc_id", "text").collect()
eval_rows = []

for d in docs:
    prompt = f"""Read this runbook and write ONE short question an engineer might type into a support chat, whose answer is in this runbook.
Use everyday words. Do NOT copy the runbook title or its exact phrases. Output ONLY the question.

Runbook:
{d['text']}"""
    q = call_llm(prompt, max_tokens=60).strip().strip('"')
    eval_rows.append((q, d["doc_id"]))
    time.sleep(0.5)

for q, doc in eval_rows[:8]:
    print(doc, "->", q)

adf_pipeline_failed_activity -> What causes a pipeline to fail with a red cross in the monitoring dashboard?
adf_trigger_missed -> What can I do if my daily data pipeline isn't running on schedule?
backfill_procedure -> What can I do if a data backfill process deleted my current data?
bad_csv_delimiter -> What causes columns to get mixed up when reading CSV files?
checkpoint_corruption -> What should I do when my streaming pipeline keeps failing due to corrupted metadata?
compute_start_failure -> What can I do if my Databricks job fails right away due to resource limits?
concurrent_write_error -> What can I do if my streaming queries keep failing due to concurrent writes?
corrupt_json_record -> How can I handle Spark jobs that fail due to bad JSON data?


In [0]:
eval_table = f"{CATALOG}.{SCHEMA}.eval_questions"
spark.createDataFrame(eval_rows, "q string, expected_doc string") \
     .write.mode("overwrite").saveAsTable(eval_table)
display(spark.table(eval_table))

q,expected_doc
What causes a pipeline to fail with a red cross in the monitoring dashboard?,adf_pipeline_failed_activity
What can I do if my daily data pipeline isn't running on schedule?,adf_trigger_missed
What can I do if a data backfill process deleted my current data?,backfill_procedure
What causes columns to get mixed up when reading CSV files?,bad_csv_delimiter
What should I do when my streaming pipeline keeps failing due to corrupted metadata?,checkpoint_corruption
What can I do if my Databricks job fails right away due to resource limits?,compute_start_failure
What can I do if my streaming queries keep failing due to concurrent writes?,concurrent_write_error
How can I handle Spark jobs that fail due to bad JSON data?,corrupt_json_record
What should I do when my pipeline fails due to bad data?,data_quality_check_failing
What can cause a Spark job to get stuck near the end or run out of memory?,data_skew


In [0]:
spark.sql(f"DELETE FROM {eval_table} WHERE expected_doc = 'put_the_doc_id_here'")

DataFrame[num_affected_rows: bigint]

In [0]:
eval_set = [(r.q, r.expected_doc) for r in spark.table(eval_table).collect()]
print(len(eval_set), "questions")

def hit_rate(index_name, k):
    hits = 0
    for q, expected in eval_set:
        found = {row[1] for row in search(q, index_name, k)}
        hits += expected in found
    return hits / len(eval_set)

results = []
for name, idx in [("small (80 words)", INDEX_A), ("large (whole runbook)", INDEX_B)]:
    for k in [1, 3, 5]:
        results.append((name, k, round(hit_rate(idx, k), 3)))

display(spark.createDataFrame(results, "index string, k int, hit_rate double"))

30 questions


index,k,hit_rate
small (80 words),1,0.933
small (80 words),3,1.0
small (80 words),5,1.0
large (whole runbook),1,0.933
large (whole runbook),3,1.0
large (whole runbook),5,1.0


In [0]:
for q, expected in eval_set:
    found = [row[1] for row in search(q, INDEX_A, 3)]
    if expected not in found:
        print("MISS:", q, "| expected:", expected, "| got:", found)